# CEG-WM experiment-execution bootstrap

This thin entrypoint only verifies and invokes a separately distributed bootstrap. The current package entrypoint is a CPU/synthetic wiring check: it is not GPU, held-out, calibration, baseline, runtime-qualification, or scientific-effect evidence.

In [ ]:
from google.colab import drive, files
from pathlib import Path
from hashlib import sha256
import json
import subprocess
import sys

drive.mount('/content/drive')

In [ ]:
DRIVE_ROOT = Path('/content/drive/MyDrive/CEG-WM')
BOOTSTRAP_SOURCE = DRIVE_ROOT / 'bootstrap/experiment_execution_bootstrap.py'
PACKAGE_ZIP = DRIVE_ROOT / 'execution_packages/current/ceg_wm_experiment_execution.zip'
EXPECTED_BOOTSTRAP_IDENTITY = 'ceg_wm_experiment_execution_bootstrap'
EXPECTED_BOOTSTRAP_SCHEMA_VERSION = '1'
EXPECTED_BOOTSTRAP_SHA256 = '9478b50187938ea73ccb489ef9381133f8c056fb698795d4beb10ce50b127633'
EXPECTED_ARCHIVE_SHA256 = 'PASTE_INDEPENDENTLY_AUDITED_ARCHIVE_SHA256'
EXPECTED_REVISION = 'PASTE_INDEPENDENTLY_AUDITED_40_HEX_REVISION'
EXPECTED_CANDIDATE_CONFIG_DIGEST = 'PASTE_INDEPENDENTLY_AUDITED_CANDIDATE_DIGEST'
EXPECTED_EXECUTION_CONFIG_DIGEST = 'PASTE_INDEPENDENTLY_AUDITED_EXECUTION_DIGEST'
EXPECTED_INPUT_MANIFEST_DIGEST = 'PASTE_INDEPENDENTLY_AUDITED_INPUT_DIGEST'
RUN_ID = 'PASTE_UNIQUE_RUN_ID'
EPHEMERAL_ROOT = Path('/content/ceg_wm_experiment_execution')
PERSISTENT_ROOT = DRIVE_ROOT / 'experiment_execution_results'

In [ ]:
bootstrap_bytes = BOOTSTRAP_SOURCE.read_bytes()
assert sha256(bootstrap_bytes).hexdigest() == EXPECTED_BOOTSTRAP_SHA256
BOOTSTRAP_SNAPSHOT = Path('/content/experiment_execution_bootstrap.py')
with BOOTSTRAP_SNAPSHOT.open('xb') as destination:
    destination.write(bootstrap_bytes)
assert sha256(BOOTSTRAP_SNAPSHOT.read_bytes()).hexdigest() == EXPECTED_BOOTSTRAP_SHA256

In [ ]:
command = [
    sys.executable, str(BOOTSTRAP_SNAPSHOT),
    '--package-zip', str(PACKAGE_ZIP),
    '--expected-archive-sha256', EXPECTED_ARCHIVE_SHA256,
    '--expected-bootstrap-identity', EXPECTED_BOOTSTRAP_IDENTITY,
    '--expected-bootstrap-schema-version', EXPECTED_BOOTSTRAP_SCHEMA_VERSION,
    '--expected-bootstrap-sha256', EXPECTED_BOOTSTRAP_SHA256,
    '--expected-revision', EXPECTED_REVISION,
    '--expected-candidate-config-digest', EXPECTED_CANDIDATE_CONFIG_DIGEST,
    '--expected-execution-config-digest', EXPECTED_EXECUTION_CONFIG_DIGEST,
    '--expected-input-manifest-digest', EXPECTED_INPUT_MANIFEST_DIGEST,
    '--ephemeral-root', str(EPHEMERAL_ROOT),
    '--persistent-root', str(PERSISTENT_ROOT),
    '--run-id', RUN_ID,
]
completed = subprocess.run(command, check=False, capture_output=True, text=True)
print(completed.stdout)
if completed.stderr:
    print(completed.stderr)
result = json.loads(completed.stdout)
assert completed.returncode in (0, 3, 4)

In [ ]:
artifact_path = result.get('result_zip') or result.get('diagnostic_zip')
assert artifact_path
print(result['artifact_kind'], artifact_path)
files.download(artifact_path)